# A Full Business Solution

Creating a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

## Setup: Website scraping utilities

This notebook merges the standalone `scraper.py` module directly into the notebook, so the
`fetch_website_contents` and `fetch_website_links` functions are defined below instead of being imported
from an external file.

- `fetch_website_contents(url)` — fetches a page and returns its title plus visible body text (truncated to 2,000 characters).
- `fetch_website_links(url)` — fetches a page and returns the list of all hyperlinks (`href` values) found on it.

In [20]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"[Could not fetch {url}: {e}]"

    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title and soup.title.string else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the website at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

## Imports

In [2]:

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

## Initialize and constants

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://devsinc.com")
links

['/',
 '#',
 '/services/website-development',
 '/services/mobile-development',
 '/services/custom-development',
 '/services/ui-ux-design',
 '/services/d365-erp',
 '/services/d365-crm',
 '/services/power-apps',
 '/services/salesforce',
 '/services/shopify',
 '/services/design-development',
 '/services/maintenance-support',
 '/services/automation-apps',
 '/services/metaverse',
 '/services/augmented-reality',
 '/services/blockchain-cryptography',
 '/services/genai',
 '/services/data-analytics-and-insights',
 '/services/staff-augmentation',
 '/services/quality-assurance',
 '/services/devops',
 '/services/cybersecurity-solutions',
 '/services/saas',
 '/services/gaming-art-design',
 '/services/web3-gaming',
 '/services/ar-vr-xr-gaming',
 '/services/cloud-application',
 '/services/cloud-migration-cloud-ops',
 '/services/cloud-maintenance-integration',
 '/industry/shopify',
 '/industry/travel-hospitality',
 '/industry/public-sector',
 '/industry/telecommunication',
 '/industry/retail-and-cpg',

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://devsinc.com"))


Here is the list of links on the website https://devsinc.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
#
/services/website-development
/services/mobile-development
/services/custom-development
/services/ui-ux-design
/services/d365-erp
/services/d365-crm
/services/power-apps
/services/salesforce
/services/shopify
/services/design-development
/services/maintenance-support
/services/automation-apps
/services/metaverse
/services/augmented-reality
/services/blockchain-cryptography
/services/genai
/services/data-analytics-and-insights
/services/staff-augmentation
/services/quality-assurance
/services/devops
/services/cybersecurity-solutions
/services/saas
/services/gaming-art-design
/services/web3-gaming
/services/ar-vr-xr-gaming
/services/cloud-application
/services/cloud-migration-cloud-ops
/services/

### Selecting relevant links with the model

Note: the final, corrected version of `select_relevant_links` (with the proper `system`/`user` roles and a
print statement showing progress) is the one that should be used going forward — it replaces the earlier
draft versions.

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://devsinc.com")

Selecting relevant links for https://devsinc.com by calling gpt-5-nano
Found 31 relevant links


{'links': [{'type': 'homepage', 'url': 'https://devsinc.com/'},
  {'type': 'about page', 'url': 'https://devsinc.com/about-us'},
  {'type': 'leadership', 'url': 'https://devsinc.com/#leadership'},
  {'type': 'geography', 'url': 'https://devsinc.com/about-us#geography'},
  {'type': 'careers page', 'url': 'https://devsinc.com/career'},
  {'type': 'ambassador program',
   'url': 'https://devsinc.com/ambassador-program'},
  {'type': 'careers portal', 'url': 'https://apply.workable.com/devsinc-17/'},
  {'type': 'contact page', 'url': 'https://devsinc.com/contact'},
  {'type': 'case studies', 'url': 'https://devsinc.com/case-studies'},
  {'type': 'case study: Interwood',
   'url': 'https://devsinc.com/case-studies/interwood'},
  {'type': 'case study: Empowering XQuic',
   'url': 'https://devsinc.com/case-studies/empowering-xquic-for-automated-financial-accuracy'},
  {'type': 'news', 'url': 'https://devsinc.com/news'},
  {'type': 'article: cloud computing for small businesses',
   'url': 'htt

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://devsinc.com"))

Selecting relevant links for https://devsinc.com by calling gpt-5-nano
Found 13 relevant links
## Landing Page:

Devsinc | Leading Software & Product Development Agency In USA

What we do
Capabilities
Digital Transformation
Web development
App Development
Custom Software Development
UX/UI Design
Business Applications
Dynamics 365 ERP
Dynamics 365 CRM
Power Apps
Salesforce
Shopify
Design & Development
Maintenance & Support
Automation & Apps
Emerging Technologies
Metaverse
Augmented reality
Blockchain & Cryptography
Gen AI
Data Analytics
Staff Augmentation
Quality Assurance
DevOps
Cybersecurity
SaaS
Gaming
Art & Design
Web3
AR/VR/XR
Cloud
Cloud Application
Cloud Ops & Migration
Cloud maintenance & integration
Who we help
Industries
Shopify
Travel & Hospitality
Public Sector
Telecommunication
Retail & CPG
Oil, Gas, and Energy
Startups
E-commerce
Banking & Fintech
Healthcare & Pharmaceuticals
Gaming
Who We Are
About
Leadership
Geographies
Awards & Recognition
Media & Investor Relations
ESG

### Brochure system prompt

The prompt below uses the humorous/witty tone. If you'd prefer a more standard, professional brochure,
swap in the commented-out alternative.

In [15]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("Devsinc", "https://devsinc.com")

Selecting relevant links for https://devsinc.com by calling gpt-5-nano
Found 20 relevant links


"\nYou are looking at a company called: Devsinc\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nDevsinc | Leading Software & Product Development Agency In USA\n\nWhat we do\nCapabilities\nDigital Transformation\nWeb development\nApp Development\nCustom Software Development\nUX/UI Design\nBusiness Applications\nDynamics 365 ERP\nDynamics 365 CRM\nPower Apps\nSalesforce\nShopify\nDesign & Development\nMaintenance & Support\nAutomation & Apps\nEmerging Technologies\nMetaverse\nAugmented reality\nBlockchain & Cryptography\nGen AI\nData Analytics\nStaff Augmentation\nQuality Assurance\nDevOps\nCybersecurity\nSaaS\nGaming\nArt & Design\nWeb3\nAR/VR/XR\nCloud\nCloud Application\nCloud Ops & Migration\nCloud maintenance & integration\nWho we help\nIndustries\nShopify\nTravel & Hospitality\nPublic Sector\nTelecommunication\nRetail & CPG\nOil, Gas, and Ene

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("Devsinc", "https://devsinc.com")

Selecting relevant links for https://devsinc.com by calling gpt-5-nano
Found 20 relevant links


# Welcome to Devsinc: Where Code Meets Creativity and AI Moves at Warp Speed!

---

## Who Are We?  
Devsinc isn’t just another tech company. Oh no. We're a vibrant, fast-growing global family of 1,200+ coding wizards, design gurus, and innovation ninjas spread across North America, MENA, Asia Pacific, Europe & UK, and beyond. We specialize in turning complex digital dreams into smooth-running realities with enterprise-grade software and products that don’t just meet expectations—they redefine them.  

We’re the folks who believe that technology should *empower* businesses and *inspire* innovation, all while having a little fun (because who said serious tech can’t be seriously cool?).

---

## What Magic Do We Cook?  
If it’s digital, we develop it. Your one-stop shop for everything from:

- **Digital Transformation** that flips your business into the future  
- **Custom Software Development** tailor-made to fit your quirks  
- **Mobile & Web App Development** – because everyone’s on the go  
- **UX/UI Design** that your users will *actually* enjoy  
- **Business Applications** like Dynamics 365, Salesforce, Shopify  
- And that’s barely scratching the surface...

We’re also diving headfirst into the newest buzzwords (and realities):

- Metaverse & Augmented Reality (putting the "reality" in "really cool tech")  
- Blockchain & Cryptography for secure stuff you don’t want your grandma to crack  
- Gen AI & Data Analytics for those “how did we live without this?” aha moments  
- DevOps, Cybersecurity, SaaS, Cloud Migration – keeping your systems humming and safe  

---

## Who Do We Hang Out With? (Our Valued Clients)  
From **startups wearing pajamas while dreaming big** to **globetrotting enterprises running banks, healthcare, oil & gas, telecom, travel, gaming, and retail**, we’ve got you covered. If you breathe innovation and love digital leaps, you’re probably one of us!  

Industries served include:  
- Banking & Fintech  
- Healthcare & Pharmaceuticals  
- Public Sector  
- Retail & Consumer Packaged Goods  
- E-commerce & Shopify Shops  
- Gaming & Entertainment – because who doesn’t want to play and profit?  

---

## Culture & Careers: Join the Fun!  
At Devsinc, work isn’t work *per se.* It’s a playground of ideas, diversity, and inclusion, where your voice matters more than your chair's comfort level (though we do have comfy chairs too).  

Perks of life at Devsinc:  
- Collaborative vibe with a sprinkle of friendly banter  
- Diversity, equity & inclusion truly baked in, not just iced on top  
- Employee success programs that make you say, *“Wow, they really do care!”*  
- Campus Ambassador programs for eager minds still hitting the books  
- Global reach but a close-knit feel (think big team, small happy family)  

Looking for a career? Explore opportunities that suit your passion and skill. Whether you’re a coding superstar, a design visionary, or a project rockstar, there’s a place for you here.  

---

## Why Choose Devsinc? Because…  

- **We’re fast** (yes, literally one of the fastest-growing tech companies around)  
- **We’re sharp** – our team is stacked with experts who love tech as much as a kid loves candy  
- **We’re versatile** – from blockchain to Shopify, AR to AI, we ride the tech spectrum like pros  
- **We listen** – your business needs + our tech prowess = a partnership made in silicon heaven  
- **We innovate** – staying ahead of the curve is the Devsinc way  

---

## Let's Build YOUR Tomorrow, Today!  

Reach out, drop us a line, and let’s talk about how Devsinc can turbo-charge your technology journey with a dash of humor and heaps of expertise. Because building at the speed of AI sounds way better with a partner who gets it.  

[Explore Careers] | [Contact Us] | [Let’s Talk Business]  

---

_Devsinc — Empowering Businesses, Inspiring Innovation, and Making Tech Awesome._

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("Devsinc", "https://devsinc.com")

Selecting relevant links for https://devsinc.com by calling gpt-5-nano
Found 15 relevant links


# Welcome to Devsinc: The Coolest Tech Wizards in Town! 🧙‍♂️🖥️

---

## Who Are We?  
Devsinc is not just a software company; we're a **rocket ship fueling digital transformation** across North America, MENA, Europe, and more! 🚀 With over **1200+ tech aficionados** on board, we craft cutting-edge software and innovative product solutions that make businesses not only run but **fly** in today's hyper-connected world.

---

## What Magic Do We Weave? 🔮

We do it all—from **digital transformation** and **custom software development** to the **metaverse** and **generative AI**. Whether it's building slick apps with seamless UX/UI, rocking **Salesforce** integrations, or diving deep into blockchain & cybersecurity trenches, Devsinc has the superpowers.

Here's a sampler platter of our spellbook:

- Web & Mobile App Development  
- Dynamics 365 ERP & CRM  
- Shopify for the e-commerce crowd  
- Augmented Reality & Virtual Reality (AR/VR/XR) adventures  
- Data Analytics that tell your story  
- DevOps, Cloud Migration & Maintenance  
- Quality Assurance & Automation

If it’s tech, chances are we've got it covered (even gaming and Web3)!

---

## Who Do We Help?  
Our clients are as diverse as a coding language menu. From **startups** to **banking & fintech**, **travel & hospitality** to **healthcare and pharma**, and even giants in **oil, gas & energy**— we’ve got their back. So if you’re a retailer or a public sector authority, buckle up; we'll upgrade your systems like pros.

---

## Life at Devsinc: Work Hard, Innovate Harder 🎉  

- **Culture:** Collaboration? Check. Innovation? Double check. Diversity, equity, and inclusion? You bet. We believe great ideas come from diverse minds and a fun, inclusive environment where everyone has a voice.  
- **Careers:** Looking to join a fast-growing global tech family? Devsinc offers ample opportunities to learn, grow, blast through career levels, and work on bleeding-edge tech. Want your code immortalized in the metaverse? Here’s your chance!  
- **Benefits & Growth:** Talent is treasured here — from campus ambassadors to seasoned pros, your success is our priority. Enjoy competitive perks, a supportive team, and projects that challenge and inspire.

---

## Why Devsinc? Because We Build at the Speed of AI ⚡  

We don’t just keep up with technology; we set the pace. We merge passion, innovation, and expertise to deliver software solutions that drive business success—smart, secure, and scalable.

---

## Want In?  

**For Customers:** Ready to transform the way your business works? Let’s talk and create future-proof digital experiences together.

**For Innovators & Dreamers:** If you want your work to matter and your career to skyrocket, Devsinc is the place to be.

**For Investors:** Back a company that's growing like a well-fed crypto wallet — rapidly, sustainably, and with style.

---

## Get in Touch  

Visit [devsinc.com](https://devsinc.com) or dial up our global squad. Whether you want the next-gen tech solution or your next-gen job, Devsinc is just a click away.

---

# Devsinc  
*Empowering businesses. Inspiring innovation. Building the future, today.*  

---

*P.S. We love coffee, code, and quirky tech jokes. Apply now and be part of the magic!* ☕🧙‍♀️✨

Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

In [ ]:
stream_brochure("Devsinc", "https://devsinc.com")

## The End